kernel : Python (Pixi)

In [ ]:
import anndata
import numpy as np
import pandas as pd
import os
import scanpy as sc
from anndata import AnnData
from scipy import sparse
from scipy.sparse import csr_matrix
from pipeline.utils.env import find_env_dir
from pipeline.utils import plot
from pipeline.utils.get_ensembl_to_symbol import get_ensg_to_symbol

anndata.settings.allow_write_nullable_strings = True

ensg_to_symbol = get_ensg_to_symbol()

series_name = "macnair"
clustered_data_location = find_env_dir("CLUSTERED_DATA")
deseq_location = find_env_dir("DESEQ")

clustered_adata = sc.read_h5ad(os.path.join(clustered_data_location, series_name + "_clustered.h5ad"))
vn = pd.Index(clustered_adata.var_names)

s = vn.to_series(index=vn)
parts = s.str.rsplit("_", n=1, expand=True) 

sym_guess = parts[0].astype("string")
ensg_raw  = parts[1].astype("string")

bad = ~ensg_raw.str.match(r"^ENSG\d+$", na=False)
if bad.any():
    print("[WARN] Non-ENSG-like var_names examples:")
    print(vn[bad][:20].tolist())

ensg = ensg_raw.str.split(".", n=1).str[0]

mapped = ensg.map(ensg_to_symbol).replace("", pd.NA).fillna(sym_guess)

clustered_adata.var["orig_var_name"] = vn.astype(str).values
clustered_adata.var_names = mapped.values.astype(str).tolist()
clustered_adata.var_names_make_unique()

de_result = pd.read_csv(os.path.join(deseq_location, series_name + "_deseq2.csv"))
vn = de_result["gene_id"].astype(str)
parts = vn.str.rsplit("_", n=1, expand=True)
sym_guess = parts[0]
ensg = parts[1].str.split(".", n=1).str[0] 

mapped = ensg.map(ensg_to_symbol).replace("", pd.NA).fillna(sym_guess)
de_result.index = mapped.values
de_result.index.name = "gene"
de_result = de_result.iloc[:, 1:] 

de_result.sort_values("log2FoldChange_shrunk", ascending=False, inplace=True)

LOG_FOLD_CHANGE_THRESHOLD = 2
ADJUSTED_PVALUE_THRESHOLD = 0.05

In [ ]:
cell_marker_genes = {
    "OPC": ["PDGFRA", "OLIG1", "OLIG2", "PCDH15", "MEGF11", "VCAN",  "PTPRZ1", "EPN2", 
            "HAS2", "FERMT1", "CACNG5", "COL9A1", "MYT1", ],
    "COP": ["BCAN", "GPR17", "TCF7L2", "BCAS1", ],
    "Oligodendrocyte": ["MBP", "MOG", "MAG", "CNP", "MOBP", "CLDN11", "FA2H", 
                        "PPP1R14A", "ERMN", "GPR37", "KLK6", "TF", "UGT8", ],
    "Neuron": ["MAP2", "MYRF", "STMN2", "CACNG3", "RBFOX3", "SNAP25", "ENO2", "SLC17A7", "SYT1", 
               "CCK", "GABRA1", "GRIN1", "SATB2", "GRIN2B","MYT1L", "SYN1", "TMEM130", ],
    "Astrocyte": ["GFAP", "AQP4", "ALDH1L1", "SOX9", "MLC1", "ATP1B2", "GJA1", 
                  "SLC14A1", "ALDH1A1", "AGT", "ATP13A4", "BMPR1B", "CHI3L1", 
                  "APOE", "GLIS3", "ID4", "RFX4", "SLC4A4", ],
    "Microglia": ["P2RY12", "TMEM119", "CX3CR1", "CSF1R", "TREM2", "ITGAM", "ITGAX", "SLC2A5", ],
    "Endothelial": ["CLDN5", "PECAM1", "CD34", "FLT1", "VWF", "CDH5", "CLEC14A", "ERG", "ITM2A", "TM4SF1", ],
    "Pericyte": ["PDGFRB", "RGS5", "CD248", "KCNJ8", "NOTCH3", ],
    "Ependymal": ["FOXJ1", "DYNLRB2", ],
    "Smooth Muscle": ["MYH11", "CNN1", "TAGLN", "ACTA2", ],
    "Lymphocyte": ["CD3D", "CD3E", "CD4", "CD8A", "NKG7", "TRAC", ],
    "Macrophage": ["CD68", "MRC1", "LYZ", ],
    "VLMC": ["COL1A2", "COL3A1", "DCN", ],
}
plot.plot_dotplot(clustered_adata, series_name, cell_marker_genes, group = "leiden")

In [ ]:
plot.plot_proportions(clustered_adata, series_name, sample_key="sample", group_key="leiden")
plot.plot_proportions(clustered_adata, series_name, sample_key="celltype", group_key="leiden")
plot.plot_proportions(clustered_adata, series_name, sample_key="diagnosis", group_key="leiden")
plot.plot_proportions(clustered_adata, series_name, sample_key="lesion", group_key="leiden")
plot.plot_proportions(clustered_adata, series_name, sample_key="matter", group_key="leiden")
plot.plot_proportions(clustered_adata, series_name, sample_key="pmi", group_key="leiden")

In [15]:
plot.plot_umap(clustered_adata, series_name, has_celltype=True)

In [ ]:
cluster = 3
subset = de_result[de_result["cluster"] == cluster]
subset = subset[(subset["log2FoldChange"] > LOG_FOLD_CHANGE_THRESHOLD) & (subset["padj"] < ADJUSTED_PVALUE_THRESHOLD)]
subset.head()

,gene_id,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,log2FoldChange_shrunk,lfcSE_shrunk,cluster_id,contrast,group_keys
gene,,,,,,,,,,,,
AC004852.2,AC004852.2_ENSG00000278254,1157.960173,9.002661,0.091544,98.342141,0.000000e+00,0.000000e+00,9.001931,0.096963,3,rest,leiden
BEST3,BEST3_ENSG00000127325,389.759339,8.643106,0.127220,67.938466,0.000000e+00,0.000000e+00,8.644003,0.139883,3,rest,leiden
LINC01546,LINC01546_ENSG00000228459,26.534939,8.168023,0.180186,45.331076,0.000000e+00,0.000000e+00,8.429485,0.198859,3,rest,leiden
FMO1-AS1,AL445673.1_ENSG00000231424,4.242455,6.972347,0.284561,24.502081,1.403608e-132,4.333106e-131,8.281955,0.381883,3,rest,leiden
PDGFRA,PDGFRA_ENSG00000134853,352.711525,8.155413,0.098939,82.428837,0.000000e+00,0.000000e+00,8.154027,0.103952,3,rest,leiden


Identify clusters where a specific gene is differentially expressed

In [64]:
gene = "HIF1A"
subset = de_result.loc[gene]
subset = subset[(subset["log2FoldChange"] > LOG_FOLD_CHANGE_THRESHOLD) & (subset["padj"] < ADJUSTED_PVALUE_THRESHOLD)]
subset.head()

,gene_id,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,log2FoldChange_shrunk,lfcSE_shrunk,cluster_id,contrast,group_keys
gene,,,,,,,,,,,,
HIF1A,HIF1A_ENSG00000100644,3506.290852,3.38924,0.160214,21.154504,2.508126e-99,7.785429e-97,3.376918,0.167114,55,rest,leiden


In [ ]:
# The ratio threshold for filtering unique markers
# The ratio is determined by (highest normalized mean count) / (second highest normalized count)
UNIQUE_THRESHOLD = 1

def filter_unique_markers(adata: AnnData, deseq_results: pd.DataFrame, cluster_key="leiden", unique_threshold=UNIQUE_THRESHOLD):
    cluster_key = deseq_results["group_keys"].iloc[0].split(",")[0]
    candidates = deseq_results[
        (deseq_results["log2FoldChange"] > LOG_FOLD_CHANGE_THRESHOLD) & 
        (deseq_results["padj"] < ADJUSTED_PVALUE_THRESHOLD)
    ].copy()
    candidates["cluster"] = candidates["cluster"].astype(str)
    valid_ids_mask = adata.var_names.isin(candidates["gene"])
    adata_needed_genes = adata[:, valid_ids_mask].to_memory()

    assert isinstance(adata.X, csr_matrix)
    library_sizes = np.asarray(adata.X.sum(axis=1)).flatten()
    library_sizes[library_sizes == 0] = 1.0
 
    cluster_means_dict = dict()
    TARGET_SUM = 1e4

    for cluster in adata.obs[cluster_key].unique():
        mask = (adata.obs[cluster_key] == cluster).values
        adata_masked = adata_needed_genes[mask]
        libs = library_sizes[mask]
        scaling_factors = TARGET_SUM / libs
        
        assert isinstance(adata_masked.X, csr_matrix)
        row_scale = sparse.diags(scaling_factors)
        X_norm = row_scale @ adata_masked.X
        mean_val = np.asarray(X_norm.mean(axis=0)).flatten()

        cluster_means_dict[str(cluster)] = mean_val
    
    cluster_means = pd.DataFrame.from_dict(cluster_means_dict, orient="index", columns=adata_needed_genes.var_names) 
    unique_markers = []

    for cluster in candidates["cluster"].unique():
        cluster_sub = candidates[candidates["cluster"] == cluster]
        genes = cluster_sub["gene"].values

        my_means = cluster_means.loc[cluster, genes].values
        rest_means: pd.DataFrame = cluster_means.drop(index=cluster)[genes].max(axis=0).values
        safe_rest_means = np.maximum(rest_means, 1e-9)
        
        ratios = my_means / safe_rest_means
        pass_mask = ratios >= unique_threshold
        
        valid_rows = cluster_sub[pass_mask].copy()
        valid_rows["specificity_ratio"] = ratios[pass_mask]
        valid_rows["my_mean"] = my_means[pass_mask]
        valid_rows["second_best_mean"] = rest_means[pass_mask]

        unique_markers.append(valid_rows)

    markers = pd.concat(unique_markers, ignore_index = True)
    markers.set_index("gene", inplace=True)
    markers.sort_values("log2FoldChange_shrunk", ascending=False, inplace=True)

    return markers

In [ ]:
markers = filter_unique_markers(clustered_adata, de_result)
markers.head()

In [ ]:
cluster_markers = markers[(markers["cluster"] == "58") & (markers["my_mean"] > 0)]
cluster_markers.head(20)

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,log2FoldChange_shrunk,lfcSE_shrunk,cluster_id,contrast,group_keys,specificity_ratio,my_mean,second_best_mean
gene,,,,,,,,,,,,,,
SPINK4,1.428653,3.527486,0.733621,4.808322,0.000002,0.000011,1.782944,0.960382,58,rest,leiden,2.651581,0.018447,0.006957
AL050343.2,1.049335,2.825082,0.649384,4.350405,0.000014,0.000083,1.269608,0.802908,58,rest,leiden,2.256483,0.011194,0.004961


In [ ]:
h5ad_matrix_location = find_env_dir("H5AD_MATRIX")
opc = clustered_adata[
    (clustered_adata.obs["leiden"] == "3") |
    (clustered_adata.obs["leiden"] == "29")
]
opc.write_h5ad(os.path.join(h5ad_matrix_location, series_name + "_opc.h5ad"))

In [ ]:
import os
import pandas as pd
import scanpy as sc
from pipeline.utils.env import find_env_dir
from pipeline.utils.find_reference_gene import find_pseudobulk_reference_genes

series_name = "macnair_opc"
clustered_data_location = find_env_dir("CLUSTERED_DATA")
deseq_location = find_env_dir("DESEQ")

opc_adata = sc.read_h5ad(os.path.join(clustered_data_location, series_name + "_clustered.h5ad"))
opc_de_result = pd.read_csv(os.path.join(deseq_location, series_name + "_deseq2.csv"), index_col=0)
opc_de_result.sort_values("log2FoldChange_shrunk", ascending=False, inplace=True)

LOG_FOLD_CHANGE_THRESHOLD = 2
ADJUSTED_PVALUE_THRESHOLD = 0.05

In [ ]:
markers = filter_unique_markers(opc_adata, de_result)
# deg = markers[(markers["cluster"] == "AL") & (markers["baseMean"] > 1)]
deg = markers[(markers["log2FoldChange"] > 2) & (markers["baseMean"] > 10)]
deg.head(20)

In [75]:
opc_reference_genes = find_pseudobulk_reference_genes(opc_adata, min_sample_fraction=0.7)
target_genes = ["ACTB", "GAPDH", "PPIA", "MYCBP2", "CTTNBP2", "FTO", "ARID1B", "FTX"]
genes_found = opc_reference_genes.index.intersection(target_genes)
result = opc_reference_genes.loc[genes_found]
print(result)

         Expressed_Fraction    Mean_CPM  Std_Deviation        CV
MYCBP2             1.000000  415.069763      53.362617  0.128563
CTTNBP2            1.000000  526.320435      73.736046  0.140097
FTO                1.000000  291.292938      40.969929  0.140649
ARID1B             1.000000  573.833557      82.553070  0.143862
FTX                1.000000  958.714966     138.133759  0.144082
ACTB               0.942308  206.344788     159.239838  0.771717
PPIA               0.858974   28.251514      24.012129  0.849941
GAPDH              0.910256  140.610550     148.293457  1.054640


In [65]:
cell_marker_genes = {
    "Conventional": ["ACTB", "GAPDH", "PPIA"],
    "Proposed": ["MYCBP2", "CTTNBP2", "FTO", "ARID1B", "FTX"]
}
plot.plot_dotplot(opc_adata, series_name, cell_marker_genes, group = "lesion")

In [ ]:
cell_marker_genes = {
    "OPC": ["PDGFRA", "OLIG1", "OLIG2", "PCDH15", "MEGF11", "VCAN",  "PTPRZ1", "EPN2", 
            "HAS2", "FERMT1", "CACNG5", "COL9A1", "MYT1", ],
    "COP": ["BCAN", "GPR17", "TCF7L2", "BCAS1", ],
    "Oligodendrocyte": ["MBP", "MOG", "MAG", "CNP", "MOBP", "CLDN11", "FA2H", 
                        "PPP1R14A", "ERMN", "GPR37", "KLK6", "TF", "UGT8", ],
    "Neuron": ["MAP2", "MYRF", "STMN2", "CACNG3", "RBFOX3", "SNAP25", "ENO2", "SLC17A7", "SYT1", 
               "CCK", "GABRA1", "GRIN1", "SATB2", "GRIN2B","MYT1L", "SYN1", "TMEM130", ],
    "Astrocyte": ["GFAP", "AQP4", "ALDH1L1", "SOX9", "MLC1", "ATP1B2", "GJA1", 
                  "SLC14A1", "ALDH1A1", "AGT", "ATP13A4", "BMPR1B", "CHI3L1", 
                  "APOE", "GLIS3", "ID4", "RFX4", "SLC4A4", ],
    "Microglia": ["P2RY12", "TMEM119", "CX3CR1", "CSF1R", "TREM2", "ITGAM", "ITGAX", "SLC2A5", ],
    "Endothelial": ["CLDN5", "PECAM1", "CD34", "FLT1", "VWF", "CDH5", "CLEC14A", "ERG", "ITM2A", "TM4SF1", ],
    "Pericyte": ["PDGFRB", "RGS5", "CD248", "KCNJ8", "NOTCH3", ],
    "Ependymal": ["FOXJ1", "DYNLRB2", ],
    "Smooth Muscle": ["MYH11", "CNN1", "TAGLN", "ACTA2", ],
    "Lymphocyte": ["CD3D", "CD3E", "CD4", "CD8A", "NKG7", "TRAC", ],
    "Macrophage": ["CD68", "MRC1", "LYZ", ],
    "VLMC": ["COL1A2", "COL3A1", "DCN", ],
}
plot.plot_dotplot(opc_adata, series_name, cell_marker_genes, group = "leiden")

/home/sjkim/protocols/.pixi/envs/default/lib/python3.14/site-packages/scanpy/plotting/_utils.py:999: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig = plt.figure(figsize=ax_or_figsize)


In [ ]:
plot.plot_proportions(opc_adata, series_name, sample_key="sample", group_key="leiden")
plot.plot_proportions(opc_adata, series_name, sample_key="celltype", group_key="leiden")
plot.plot_proportions(opc_adata, series_name, sample_key="diagnosis", group_key="leiden")
plot.plot_proportions(opc_adata, series_name, sample_key="lesion", group_key="leiden")
plot.plot_proportions(opc_adata, series_name, sample_key="matter", group_key="leiden")
plot.plot_proportions(opc_adata, series_name, sample_key="pmi", group_key="leiden")

In [ ]:
import anndata
import numpy as np
import pandas as pd
import os
import scanpy as sc
from anndata import AnnData
from scipy import sparse
from scipy.sparse import csr_matrix
from pipeline.utils.env import find_env_dir
from pipeline.utils import plot
from pipeline.utils.get_ensembl_to_symbol import get_ensg_to_symbol

anndata.settings.allow_write_nullable_strings = True

ensg_to_symbol = get_ensg_to_symbol()

series_name = "macnair_validation"
clustered_data_location = find_env_dir("CLUSTERED_DATA")
deseq_location = find_env_dir("DESEQ")

validation_adata = sc.read_h5ad(os.path.join(clustered_data_location, series_name + "_clustered.h5ad"))
vn = pd.Index(validation_adata.var_names)

s = vn.to_series(index=vn)
parts = s.str.rsplit("_", n=1, expand=True) 

sym_guess = parts[0].astype("string")
ensg_raw  = parts[1].astype("string")

bad = ~ensg_raw.str.match(r"^ENSG\d+$", na=False)
if bad.any():
    print("[WARN] Non-ENSG-like var_names examples:")
    print(vn[bad][:20].tolist())

ensg = ensg_raw.str.split(".", n=1).str[0]

mapped = pd.Series([ensg_to_symbol.get(x) for x in ensg], index=vn)
mapped = mapped.replace("", pd.NA)
final_symbol = mapped.fillna(pd.Series(sym_guess.values, index=vn))

validation_adata.var["orig_var_name"] = vn.astype(str).values
validation_adata.var_names = final_symbol.values.astype(str).tolist()
validation_adata.var_names_make_unique()

de_result = pd.read_csv(os.path.join(deseq_location, series_name + "_deseq2.csv"))
vn = de_result["gene_id"].astype(str)
parts = vn.str.rsplit("_", n=1, expand=True)
sym_guess = parts[0]
ensg = parts[1].str.split(".", n=1).str[0] 

mapped = ensg.map(ensg_to_symbol).fillna(sym_guess)
de_result.index = mapped.values
de_result.index.name = "gene"
de_result = de_result.iloc[:, 1:] 

de_result.sort_values("log2FoldChange_shrunk", ascending=False, inplace=True)

LOG_FOLD_CHANGE_THRESHOLD = 2
ADJUSTED_PVALUE_THRESHOLD = 0.05

In [ ]:
cell_marker_genes = {
    "OPC": ["PDGFRA", "OLIG1", "OLIG2", "PCDH15", "MEGF11", "VCAN",  "PTPRZ1", "EPN2", 
            "HAS2", "FERMT1", "CACNG5", "COL9A1", "MYT1", ],
    "COP": ["BCAN", "GPR17", "TCF7L2", "BCAS1", ],
    "Oligodendrocyte": ["MBP", "MOG", "MAG", "CNP", "MOBP", "CLDN11", "FA2H", 
                        "PPP1R14A", "ERMN", "GPR37", "KLK6", "TF", "UGT8", ],
    "Neuron": ["MAP2", "MYRF", "STMN2", "CACNG3", "RBFOX3", "SNAP25", "ENO2", "SLC17A7", "SYT1", 
               "CCK", "GABRA1", "GRIN1", "SATB2", "GRIN2B","MYT1L", "SYN1", "TMEM130", ],
    "Astrocyte": ["GFAP", "AQP4", "ALDH1L1", "SOX9", "MLC1", "ATP1B2", "GJA1", 
                  "SLC14A1", "ALDH1A1", "AGT", "ATP13A4", "BMPR1B", "CHI3L1", 
                  "APOE", "GLIS3", "ID4", "RFX4", "SLC4A4", ],
    "Microglia": ["P2RY12", "TMEM119", "CX3CR1", "CSF1R", "TREM2", "ITGAM", "ITGAX", "SLC2A5", ],
    "Endothelial": ["CLDN5", "PECAM1", "CD34", "FLT1", "VWF", "CDH5", "CLEC14A", "ERG", "ITM2A", "TM4SF1", ],
    "Pericyte": ["PDGFRB", "RGS5", "CD248", "KCNJ8", "NOTCH3", ],
    "Ependymal": ["FOXJ1", "DYNLRB2", ],
    "Smooth Muscle": ["MYH11", "CNN1", "TAGLN", "ACTA2", ],
    "Lymphocyte": ["CD3D", "CD3E", "CD4", "CD8A", "NKG7", "TRAC", ],
    "Macrophage": ["CD68", "MRC1", "LYZ", ],
    "VLMC": ["COL1A2", "COL3A1", "DCN", ],
}

plot.plot_dotplot(validation_adata, series_name, cell_marker_genes, group = "leiden")

In [131]:
plot.plot_umap(validation_adata, series_name, has_celltype=True)

In [132]:
essential_cols = ["lesion", "source"]
valid_cells_mask = ~validation_adata.obs[essential_cols].isna().any(axis=1) #type: ignore
validation_adata = validation_adata[valid_cells_mask]

h5ad_matrix_location = find_env_dir("H5AD_MATRIX")
opc = validation_adata[
    (validation_adata.obs["leiden"] == "9") |
    (validation_adata.obs["leiden"] == "29")
]
opc.write_h5ad(os.path.join(h5ad_matrix_location, series_name + "_opc.h5ad"))

In [ ]:
import os
import pandas as pd
import scanpy as sc
from pipeline.utils.env import find_env_dir
from pipeline.utils.find_reference_gene import find_pseudobulk_reference_genes

anndata.settings.allow_write_nullable_strings = True

ensg_to_symbol = get_ensg_to_symbol()

series_name = "macnair_validation_opc"
clustered_data_location = find_env_dir("CLUSTERED_DATA")
deseq_location = find_env_dir("DESEQ")

validation_opc_adata = sc.read_h5ad(os.path.join(clustered_data_location, series_name + "_clustered.h5ad"))

validation_opc_de_result = pd.read_csv(os.path.join(deseq_location, series_name + "_deseq2.csv"), index_col=0)
validation_opc_de_result.sort_values("log2FoldChange_shrunk", ascending=False, inplace=True)

LOG_FOLD_CHANGE_THRESHOLD = 2
ADJUSTED_PVALUE_THRESHOLD = 0.05

In [79]:
validation_opc_reference_genes = find_pseudobulk_reference_genes(validation_opc_adata, min_sample_fraction=0.7)
target_genes = ["ACTB", "GAPDH", "PPIA", "MYCBP2", "CTTNBP2", "FTO", "ARID1B", "FTX"]
genes_found = validation_opc_reference_genes.index.intersection(target_genes)
result = validation_opc_reference_genes.loc[genes_found]
print(result)

         Expressed_Fraction    Mean_CPM  Std_Deviation        CV
ARID1B             1.000000  583.711121      70.988907  0.121617
MYCBP2             1.000000  430.824554      59.124302  0.137235
CTTNBP2            1.000000  527.759827      81.733269  0.154868
FTO                1.000000  299.062103      56.503082  0.188934
FTX                1.000000  987.951477     193.265610  0.195623
ACTB               0.925926  143.608673     118.238609  0.823339
PPIA               0.759259   27.009024      27.567381  1.020673
GAPDH              0.925926  121.395416     166.853394  1.374462


In [ ]:
cell_marker_genes = {
    "Conventional": ["ACTB", "GAPDH", "PPIA"],
    "Proposed": ["MYCBP2", "CTTNBP2", "FTO", "ARID1B", "FTX"]
}
plot.plot_dotplot(validation_opc_adata, series_name, cell_marker_genes, group = "lesion")

In [58]:
validation_opc_de_result[validation_opc_de_result["baseMean"] > 100].head(20)

,gene,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,log2FoldChange_shrunk,lfcSE_shrunk,cluster,contrast,group_keys
52129,MMD2,122.036834,0.888703,0.248148,3.581339,0.000342,0.064545,0.722296,0.267181,WM,rest,lesion
48964,TNK2,215.766678,0.750954,0.211116,3.557071,0.000375,0.065567,0.602711,0.231491,WM,rest,lesion
46608,AC132153.1,139.921919,0.666231,0.174076,3.827237,0.000130,0.047795,0.558324,0.186110,WM,rest,lesion
50308,SERINC5,205.837933,0.517631,0.116656,4.437246,0.000009,0.008111,0.462056,0.119543,WM,rest,lesion
78483,ARHGAP42,197.761840,0.535801,0.156180,3.430664,0.000602,0.192244,0.421820,0.184662,NAWM,rest,lesion
61338,PLLP,102.463415,0.623532,0.224198,2.781174,0.005416,0.178732,0.385645,0.304505,WM,rest,lesion
64201,OLFM2,118.325685,0.457247,0.123296,3.708535,0.000208,0.051957,0.379563,0.132320,WM,rest,lesion
137,CLSTN1,107.476797,0.481843,0.141588,3.403141,0.000666,0.707491,0.375536,0.165245,CIL,rest,lesion
64044,ATCAY,107.406288,0.502247,0.162764,3.085728,0.002031,0.128417,0.361588,0.192444,WM,rest,lesion
79491,ATRNL1,1142.263917,0.545661,0.184891,2.951253,0.003165,0.270053,0.358660,0.285695,NAWM,rest,lesion


In [44]:
validation_opc_markers = filter_unique_markers(validation_opc_adata, validation_opc_de_result, cluster_key="lesion", unique_threshold=0)
validation_opc_markers.head()

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,log2FoldChange_shrunk,lfcSE_shrunk,cluster,contrast,group_keys,specificity_ratio,my_mean,second_best_mean
gene,,,,,,,,,,,,,,
PAPPA,21.614521,2.699466,0.567624,4.755729,0.000002,0.04357,2.32839,0.676012,CIL,rest,lesion,0.897694,0.558657,0.622324
